# OSHA Injury Data Analysis (2020–2023)

This section presents three scalable and insightful visualizations based on the OSHA Injury Tracking Application dataset (2020–2023).

### Overview of Visualizations:
1. **Choropleth Map** – Injury Rate per 1,000 Employees by U.S. State
2. **Bar Chart** – Injury rate per 1,000 employees across industries.
3. **Line Chart** – Year-over-year injury trends for the top 5 most affected industries.

All data cleaning, transformations, and aggregations were performed using SparkSQL, ensuring scalability for large datasets. Visualizations were created directly within Databricks notebooks using its native charting capabilities, making the analysis easy to reproduce and share.

In [0]:
# Read the CSV file
df = spark.read.csv("/FileStore/tables/OSHA_Injury_Tracking_Application_2020_2023-3.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("osha_data")
#display(df)

In [0]:
# Visualization 1: Injury Rate per 1,000 Employees by U.S. State

# Clean and filter valid U.S states abbreviations
# Aggregates total injuries and employee counts for each state
# Calculates injury rates per 1000 employees
spark.sql("""
    SELECT 
        UPPER(TRIM(state)) AS State,
        SUM(total_injuries) AS TotalInjuries,
        SUM(annual_average_employees) AS TotalEmployees,
        (SUM(total_injuries) / SUM(annual_average_employees)) * 1000 AS InjuryRatePer1000
    FROM osha_data
    WHERE 
        UPPER(TRIM(state)) IN (
            'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN',
            'IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV',
            'NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN',
            'TX','UT','VT','VA','WA','WV','WI','WY'
        )
        AND total_injuries IS NOT NULL
        AND annual_average_employees IS NOT NULL
        AND annual_average_employees > 0
    GROUP BY UPPER(TRIM(state))
""").createOrReplaceTempView("injuries_by_state")

# Display the aggregated results in a Databricks table
display(spark.sql("SELECT * FROM injuries_by_state ORDER BY TotalInjuries DESC"))


State,TotalInjuries,TotalEmployees,InjuryRatePer1000
CA,657165.0,4.3309959E7,15.173530873118581
TX,323886.0,8.9224439E7,3.6300144179107696
FL,276174.0,1.65173734E8,1.6720212912302388
PA,240712.0,6.8385722E7,3.5199160432933647
IL,234287.0,1.93458514E8,1.2110451752978937
NY,214599.0,1.05372E7,20.365846714497213
OH,189387.0,1.0778804E7,17.57031670675151
MI,170134.0,6155746.0,27.638242383620117
NC,148538.0,7.964826E7,1.8649246072670012
GA,145286.0,2.15024235E8,0.675672674756871


Databricks visualization. Run in Databricks to view.

In [0]:
# Visualization 2: Top 5 Industries by Injury Rate per 1,000 Employees

# Aggregate injury and employee by industry
# Calculates injury rate per 1000 employees.
# Returns the top 5 industries with the highest injury rates
spark.sql("""
    SELECT 
        UPPER(TRIM(`Industry Name`)) AS Industry,
        SUM(total_injuries) AS TotalInjuries,
        SUM(annual_average_employees) AS TotalEmployees,
        (SUM(total_injuries) / SUM(annual_average_employees)) * 1000 AS InjuryRatePer1000
    FROM osha_data
    WHERE 
        `Industry Name` IS NOT NULL AND
        total_injuries IS NOT NULL AND
        annual_average_employees IS NOT NULL AND
        annual_average_employees > 0
    GROUP BY UPPER(TRIM(`Industry Name`))
    ORDER BY InjuryRatePer1000 DESC
    LIMIT 5
""").createOrReplaceTempView("top_industries_rate")

# Display the top 5 industries with the highest injury rates
display(spark.sql("SELECT * FROM top_industries_rate ORDER BY InjuryRatePer1000 DESC"))

Industry,TotalInjuries,TotalEmployees,InjuryRatePer1000
"AGRICULTURE, FORESTRY, FISHING AND HUNTING",69786.0,2274555.0,30.681166206137025
RETAIL TRADE,785293.0,3.3551502E7,23.405598950532827
ACCOMMODATION AND FOOD SERVICES,122806.0,8071299.0,15.215146905101646
EDUCATIONAL SERVICES,61130.0,4357408.0,14.02898236749921
UTILITIES,41313.0,3728352.0,11.080767052038004


Databricks visualization. Run in Databricks to view.

In [0]:
# Visualization 3: Trend Over Time for Top 5 Industries

# Identify top 5 industries by total reported injuries
spark.sql("""
    SELECT 
        UPPER(TRIM(`Industry Name`)) AS Industry,
        SUM(total_injuries) AS TotalInjuries
    FROM osha_data
    GROUP BY UPPER(TRIM(`Industry Name`))
    ORDER BY TotalInjuries DESC
    LIMIT 5
""").createOrReplaceTempView("top5_industries")

# Calculates yearly injury trends for only the top 5 industries
spark.sql("""
    SELECT 
        CAST(year_filing_for AS STRING) AS Year,
        UPPER(TRIM(`Industry Name`)) AS Industry,
        SUM(total_injuries) AS TotalInjuries
    FROM osha_data
    WHERE CAST(year_filing_for AS INT) BETWEEN 2020 AND 2023
        AND UPPER(TRIM(`Industry Name`)) IN (SELECT Industry FROM top5_industries)
    GROUP BY CAST(year_filing_for AS STRING), UPPER(TRIM(`Industry Name`))
    ORDER BY Year
""").createOrReplaceTempView("industry_trends")

# Display the yearly injury trends
display(spark.sql("SELECT * FROM industry_trends ORDER BY Year, Industry"))

Year,Industry,TotalInjuries
2020,CONSTRUCTION,59042.0
2020,HEALTH CARE AND SOCIAL ASSISTANCE,250452.0
2020,MANUFACTURING,202541.0
2020,RETAIL TRADE,175172.0
2020,TRANSPORTATION AND WAREHOUSING,183039.0
2021,CONSTRUCTION,63885.0
2021,HEALTH CARE AND SOCIAL ASSISTANCE,257923.0
2021,MANUFACTURING,232902.0
2021,RETAIL TRADE,199084.0
2021,TRANSPORTATION AND WAREHOUSING,221170.0


Databricks visualization. Run in Databricks to view.